# CatBoost Model — S6E4 Irrigation Need
Mirrors the XGBoost notebook structure. Explores 3 hyperparameter sets, then trains a best model and generates a Kaggle submission.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report
from catboost import CatBoostClassifier

# Load data
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

X = train_df.drop(columns=['id', 'Irrigation_Need'])
y_raw = train_df['Irrigation_Need']

# Encode target (High=0, Low=1, Medium=2 after alphabetical sort by LabelEncoder)
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
print('Classes:', target_encoder.classes_)

# Keep categorical columns as strings — CatBoost handles them natively
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {cat_cols}')

# Train / validation split (stratified, same seed as XGBoost notebook)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape}, Validation set: {X_val.shape}')

Classes: ['High' 'Low' 'Medium']
Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


C:\Users\sydne\AppData\Local\Temp\ipykernel_31168\4039513937.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


Training set: (504000, 19), Validation set: (126000, 19)


## Hyperparameter Exploration

Three deliberately different configurations to observe how each parameter affects balanced accuracy:

| Set | Key changes | Expected effect |
|-----|-------------|----------------|
| 1 | Shallow tree (depth=4), high LR, few iterations | Fast, may underfit |
| 2 | Medium depth (depth=7), moderate LR, more iterations, L2 reg | Balanced baseline |
| 3 | Deep tree (depth=10), low LR, many iterations, high reg | Slower, may overfit or generalise better |

In [2]:
# cat_features index list required by CatBoost
cat_feature_indices = [X.columns.get_loc(c) for c in cat_cols]

# Three hyperparameter sets — each meaningfully different
catboost_param_sets = [
    # Set 1: shallow & fast — high learning rate, few iterations, no extra regularisation
    {
        'iterations': 100,
        'learning_rate': 0.15,
        'depth': 4,
        'l2_leaf_reg': 1,
        'auto_class_weights': 'Balanced',
    },
    # Set 2: medium complexity — moderate LR, border_count for finer splits
    {
        'iterations': 300,
        'learning_rate': 0.07,
        'depth': 7,
        'l2_leaf_reg': 5,
        'border_count': 64,
        'auto_class_weights': 'Balanced',
    },
    # Set 3: deep & slow — low LR, high depth, strong regularisation, random_strength for noise
    {
        'iterations': 500,
        'learning_rate': 0.03,
        'depth': 10,
        'l2_leaf_reg': 10,
        'random_strength': 3.0,
        'border_count': 128,
        'auto_class_weights': 'Balanced',
    },
]

print('=== CatBoost Hyperparameter Exploration ===')
cb_results = []
for idx, params in enumerate(catboost_param_sets, start=1):
    model = CatBoostClassifier(random_state=42, verbose=0, **params)
    model.fit(X_train, y_train, cat_features=cat_feature_indices)
    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    cb_results.append((idx, score, params, model))
    print(f'Set {idx}: balanced_accuracy = {score:.4f} | iterations={params["iterations"]}, '
          f'lr={params["learning_rate"]}, depth={params["depth"]}, l2={params["l2_leaf_reg"]}')

best_cb_idx, best_cb_score, best_cb_params, best_cb_model = max(cb_results, key=lambda x: x[1])
print(f'\nBest set: {best_cb_idx} with balanced_accuracy = {best_cb_score:.4f}')

=== CatBoost Hyperparameter Exploration ===
Set 1: balanced_accuracy = 0.9670 | iterations=100, lr=0.15, depth=4, l2=1
Set 2: balanced_accuracy = 0.9665 | iterations=300, lr=0.07, depth=7, l2=5
Set 3: balanced_accuracy = 0.9658 | iterations=500, lr=0.03, depth=10, l2=10

Best set: 1 with balanced_accuracy = 0.9670


In [3]:
# Detailed classification report for the best set
print('--- Classification Report (Best CatBoost Set) ---')
best_preds = best_cb_model.predict(X_val)
print(classification_report(y_val, best_preds, target_names=target_encoder.classes_))

--- Classification Report (Best CatBoost Set) ---
              precision    recall  f1-score   support

        High       0.85      0.94      0.89      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.96      0.97     47815

    accuracy                           0.98    126000
   macro avg       0.94      0.97      0.95    126000
weighted avg       0.98      0.98      0.98    126000



## Final Model — Retrain on Full Training Data & Generate Submission

In [4]:
print('--- Training Final CatBoost Model on Full Dataset ---')

# Rebuild cat_feature_indices on full X (same columns, same order)
cat_feature_indices_full = [X.columns.get_loc(c) for c in cat_cols]

final_cb_model = CatBoostClassifier(random_state=42, verbose=0, **best_cb_params)
final_cb_model.fit(X, y, cat_features=cat_feature_indices_full)

# Prepare test set — CatBoost handles string categoricals natively, no encoding needed
X_test = test_df.drop(columns=['id'])

cb_preds_encoded = final_cb_model.predict(X_test)
cb_preds_text = target_encoder.inverse_transform(cb_preds_encoded.astype(int))

submission_cb = pd.DataFrame({
    'id': test_df['id'],
    'Irrigation_Need': cb_preds_text
})
submission_cb.to_csv('S6E4_catboost_submission.csv', index=False)
print('Success! CatBoost submission saved to S6E4_catboost_submission.csv')
print(submission_cb['Irrigation_Need'].value_counts())

--- Training Final CatBoost Model on Full Dataset ---


c:\Users\sydne\OneDrive\Desktop\Advanced ML\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Success! CatBoost submission saved to S6E4_catboost_submission.csv
Irrigation_Need
Low       159598
Medium    100491
High        9911
Name: count, dtype: int64
